<a href="https://colab.research.google.com/github/harsha123e/saas-customer-analytics-warehouse/blob/main/saas_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q duckdb

In [ ]:
import sqlite3
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_DIR = Path("/content/saas_analytics")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

APP_DB = PROJECT_DIR / "app_activity.db"
BILLING_DB = PROJECT_DIR / "billing.db"
WAREHOUSE_DB = PROJECT_DIR / "analytics.duckdb"

for db_file in [APP_DB, BILLING_DB, WAREHOUSE_DB]:
    if db_file.exists():
        db_file.unlink()

print("Project workspace created:", PROJECT_DIR)

Project workspace created: /content/saas_analytics


In [ ]:
with sqlite3.connect(APP_DB) as conn:
    conn.executescript("""
        CREATE TABLE accounts (
            account_id INTEGER PRIMARY KEY,
            account_name TEXT,
            industry TEXT,
            created_at TEXT
        );

        CREATE TABLE users (
            user_id INTEGER PRIMARY KEY,
            account_id INTEGER,
            email TEXT,
            created_at TEXT
        );

        CREATE TABLE usage_events (
            event_id INTEGER PRIMARY KEY,
            user_id INTEGER,
            event_timestamp TEXT,
            feature_name TEXT,
            event_type TEXT,
            quantity INTEGER
        );
    """)

with sqlite3.connect(BILLING_DB) as conn:
    conn.executescript("""
        CREATE TABLE plans (
            plan_id INTEGER PRIMARY KEY,
            plan_name TEXT,
            monthly_price REAL
        );

        CREATE TABLE subscriptions (
            subscription_id INTEGER PRIMARY KEY,
            account_id INTEGER,
            plan_id INTEGER,
            start_date TEXT,
            end_date TEXT,
            status TEXT
        );

        CREATE TABLE invoices (
            invoice_id INTEGER PRIMARY KEY,
            subscription_id INTEGER,
            invoice_date TEXT,
            amount REAL,
            payment_status TEXT,
            paid_at TEXT
        );
    """)

print("Application and billing database schemas created.")

Application and billing database schemas created.


In [ ]:
np.random.seed(42)

n_accounts = 500
n_users = 3000

accounts = pd.DataFrame({
    "account_id": range(1, n_accounts + 1),
    "account_name": [f"Company_{i:04d}" for i in range(1, n_accounts + 1)],
    "industry": np.random.choice(
        ["Technology", "Finance", "Healthcare", "Retail", "Education"],
        n_accounts
    ),
    "created_at": pd.to_datetime(
        np.random.choice(
            pd.date_range("2024-01-01", "2025-12-31"),
            n_accounts
        )
    ).strftime("%Y-%m-%d")
})

users = pd.DataFrame({
    "user_id": range(1, n_users + 1),
    "account_id": np.random.randint(1, n_accounts + 1, n_users),
    "email": [f"user{i}@example.com" for i in range(1, n_users + 1)],
    "created_at": pd.to_datetime(
        np.random.choice(
            pd.date_range("2024-01-01", "2026-01-01"),
            n_users
        )
    ).strftime("%Y-%m-%d")
})

with sqlite3.connect(APP_DB) as conn:
    accounts.to_sql("accounts", conn, if_exists="append", index=False)
    users.to_sql("users", conn, if_exists="append", index=False)

print("Application database populated.")
print("Accounts:", len(accounts))
print("Users:", len(users))

display(accounts.head())
display(users.head())

Application database populated.
Accounts: 500
Users: 3000


,account_id,account_name,industry,created_at
0,1,Company_0001,Retail,2024-03-02
1,2,Company_0002,Education,2025-08-18
2,3,Company_0003,Healthcare,2025-12-29
3,4,Company_0004,Education,2024-12-07
4,5,Company_0005,Education,2025-01-31


,user_id,account_id,email,created_at
0,1,67,user1@example.com,2025-09-12
1,2,391,user2@example.com,2025-02-27
2,3,179,user3@example.com,2024-05-21
3,4,456,user4@example.com,2024-10-17
4,5,42,user5@example.com,2025-03-07


In [ ]:
n_events = 50_000

usage_events = pd.DataFrame({
    "event_id": range(1, n_events + 1),
    "user_id": np.random.randint(1, n_users + 1, n_events),
    "event_timestamp": pd.to_datetime(
        np.random.choice(
            pd.date_range("2026-01-01", "2026-06-30"),
            n_events
        )
    ),
    "feature_name": np.random.choice(
        ["Dashboard", "Reports", "Projects", "File Upload", "Collaboration"],
        n_events,
        p=[0.20, 0.20, 0.25, 0.15, 0.20]
    ),
    "event_type": np.random.choice(
        ["LOGIN", "CREATE", "UPDATE", "VIEW"],
        n_events,
        p=[0.20, 0.30, 0.15, 0.35]
    ),
    "quantity": np.random.randint(1, 11, n_events)
})

usage_events["event_timestamp"] = (
    usage_events["event_timestamp"].dt.strftime("%Y-%m-%d %H:%M:%S")
)

with sqlite3.connect(APP_DB) as conn:
    usage_events.to_sql("usage_events", conn, if_exists="append", index=False)

print("Product usage events generated.")
print("Total events:", len(usage_events))

display(usage_events.head())

Product usage events generated.
Total events: 50000


,event_id,user_id,event_timestamp,feature_name,event_type,quantity
0,1,2937,2026-06-06 00:00:00,Collaboration,UPDATE,6
1,2,2219,2026-01-29 00:00:00,Projects,CREATE,10
2,3,134,2026-03-10 00:00:00,File Upload,CREATE,4
3,4,2031,2026-01-11 00:00:00,Projects,VIEW,10
4,5,1451,2026-05-10 00:00:00,Collaboration,CREATE,7


In [ ]:
plans = pd.DataFrame({
    "plan_id": [1, 2, 3],
    "plan_name": ["Basic", "Professional", "Enterprise"],
    "monthly_price": [499, 1499, 3999]
})

with sqlite3.connect(BILLING_DB) as conn:
    plans.to_sql("plans", conn, if_exists="append", index=False)

print("Subscription plans created.")
display(plans)

Subscription plans created.


,plan_id,plan_name,monthly_price
0,1,Basic,499
1,2,Professional,1499
2,3,Enterprise,3999


In [ ]:
n_subscriptions = 500

subscriptions = pd.DataFrame({
    "subscription_id": range(1, n_subscriptions + 1),
    "account_id": range(1, n_subscriptions + 1),
    "plan_id": np.random.choice(
        [1, 2, 3],
        n_subscriptions,
        p=[0.40, 0.40, 0.20]
    ),
    "start_date": pd.to_datetime(
        np.random.choice(
            pd.date_range("2025-07-01", "2026-06-30"),
            n_subscriptions
        )
    ),
    "status": np.random.choice(
        ["ACTIVE", "CANCELLED", "PAST_DUE"],
        n_subscriptions,
        p=[0.80, 0.12, 0.08]
    )
})

end_dates = (
    subscriptions["start_date"]
    + pd.to_timedelta(
        np.random.randint(30, 180, n_subscriptions),
        unit="D"
    )
)

subscriptions["end_date"] = end_dates.where(
    subscriptions["status"] == "CANCELLED"
)

subscriptions["start_date"] = (
    subscriptions["start_date"].dt.strftime("%Y-%m-%d")
)

subscriptions["end_date"] = (
    subscriptions["end_date"].dt.strftime("%Y-%m-%d")
)

with sqlite3.connect(BILLING_DB) as conn:
    subscriptions.to_sql("subscriptions", conn, if_exists="append", index=False)

print("Customer subscriptions generated.")
print("Total subscriptions:", len(subscriptions))
print("\nSubscription status:")
print(subscriptions["status"].value_counts())

display(subscriptions.head())

Customer subscriptions generated.
Total subscriptions: 500

Subscription status:
status
ACTIVE       418
CANCELLED     49
PAST_DUE      33
Name: count, dtype: int64


,subscription_id,account_id,plan_id,start_date,status,end_date
0,1,1,3,2025-08-28,ACTIVE,NaN
1,2,2,1,2025-11-04,PAST_DUE,NaN
2,3,3,1,2025-11-02,ACTIVE,NaN
3,4,4,1,2026-02-05,ACTIVE,NaN
4,5,5,2,2025-08-05,ACTIVE,NaN


In [ ]:
plan_prices = plans.set_index("plan_id")["monthly_price"].to_dict()

snapshot_date = pd.Timestamp("2026-06-30")
invoice_records = []

invoice_id = 1

for sub in subscriptions.itertuples(index=False):
    start_date = pd.Timestamp(sub.start_date)

    end_date = (
        pd.Timestamp(sub.end_date)
        if pd.notna(sub.end_date)
        else snapshot_date
    )

    end_date = min(end_date, snapshot_date)

    if start_date > snapshot_date or end_date < start_date:
        continue

    invoice_dates = pd.date_range(
        start=start_date,
        end=end_date,
        freq="MS"
    )

    if len(invoice_dates) == 0:
        invoice_dates = [start_date]

    for invoice_date in invoice_dates:
        if invoice_date < start_date:
            invoice_date = start_date

        payment_status = np.random.choice(
            ["PAID", "FAILED", "PENDING"],
            p=[0.88, 0.08, 0.04]
        )

        paid_at = None

        if payment_status == "PAID":
            paid_at = (
                invoice_date + pd.Timedelta(days=np.random.randint(1, 6))
            )

            if paid_at > snapshot_date:
                paid_at = snapshot_date

        invoice_records.append({
            "invoice_id": invoice_id,
            "subscription_id": sub.subscription_id,
            "invoice_date": invoice_date.strftime("%Y-%m-%d"),
            "amount": plan_prices[sub.plan_id],
            "payment_status": payment_status,
            "paid_at": (
                paid_at.strftime("%Y-%m-%d")
                if paid_at is not None else None
            )
        })

        invoice_id += 1

invoices = pd.DataFrame(invoice_records)

with sqlite3.connect(BILLING_DB) as conn:
    invoices.to_sql("invoices", conn, if_exists="append", index=False)

print("Invoices generated:", len(invoices))
print("\nPayment status:")
print(invoices["payment_status"].value_counts())

display(invoices.head())

Invoices generated: 2749

Payment status:
payment_status
PAID       2425
FAILED      207
PENDING     117
Name: count, dtype: int64


,invoice_id,subscription_id,invoice_date,amount,payment_status,paid_at
0,1,1,2025-09-01,3999,PAID,2025-09-05
1,2,1,2025-10-01,3999,PAID,2025-10-05
2,3,1,2025-11-01,3999,PAID,2025-11-02
3,4,1,2025-12-01,3999,PAID,2025-12-03
4,5,1,2026-01-01,3999,PAID,2026-01-06


In [ ]:
with sqlite3.connect(APP_DB) as conn:
    app_data = {
        table: pd.read_sql_query(f"SELECT * FROM {table}", conn)
        for table in ["accounts", "users", "usage_events"]
    }

with sqlite3.connect(BILLING_DB) as conn:
    billing_data = {
        table: pd.read_sql_query(f"SELECT * FROM {table}", conn)
        for table in ["plans", "subscriptions", "invoices"]
    }

source_data = {**app_data, **billing_data}

for table, df in source_data.items():
    print(f"{table}: {len(df):,} records")

accounts: 500 records
users: 3,000 records
usage_events: 50,000 records
plans: 3 records
subscriptions: 500 records
invoices: 2,749 records


In [ ]:
warehouse = duckdb.connect(str(WAREHOUSE_DB))

warehouse.execute("CREATE SCHEMA IF NOT EXISTS raw")

for table, df in source_data.items():
    dataframe_name = f"{table}_df"

    warehouse.register(dataframe_name, df)

    warehouse.execute(f"""
        CREATE OR REPLACE TABLE raw.{table} AS
        SELECT * FROM {dataframe_name}
    """)

    warehouse.unregister(dataframe_name)

print("Raw data loaded into DuckDB.")

for table in source_data:
    count = warehouse.execute(
        f"SELECT COUNT(*) FROM raw.{table}"
    ).fetchone()[0]

    print(f"raw.{table}: {count:,} records")

Raw data loaded into DuckDB.
raw.accounts: 500 records
raw.users: 3,000 records
raw.usage_events: 50,000 records
raw.plans: 3 records
raw.subscriptions: 500 records
raw.invoices: 2,749 records


In [ ]:
quality_checks = warehouse.execute("""
    SELECT 'accounts' AS check_name,
           COUNT(*) - COUNT(DISTINCT account_id) AS issue_count
    FROM raw.accounts

    UNION ALL

    SELECT 'users.account_id -> accounts',
           COUNT(*)
    FROM raw.users u
    LEFT JOIN raw.accounts a ON u.account_id = a.account_id
    WHERE a.account_id IS NULL

    UNION ALL

    SELECT 'usage_events.user_id -> users',
           COUNT(*)
    FROM raw.usage_events e
    LEFT JOIN raw.users u ON e.user_id = u.user_id
    WHERE u.user_id IS NULL

    UNION ALL

    SELECT 'subscriptions.account_id -> accounts',
           COUNT(*)
    FROM raw.subscriptions s
    LEFT JOIN raw.accounts a ON s.account_id = a.account_id
    WHERE a.account_id IS NULL

    UNION ALL

    SELECT 'subscriptions.plan_id -> plans',
           COUNT(*)
    FROM raw.subscriptions s
    LEFT JOIN raw.plans p ON s.plan_id = p.plan_id
    WHERE p.plan_id IS NULL

    UNION ALL

    SELECT 'invoices.subscription_id -> subscriptions',
           COUNT(*)
    FROM raw.invoices i
    LEFT JOIN raw.subscriptions s
        ON i.subscription_id = s.subscription_id
    WHERE s.subscription_id IS NULL
""").df()

display(quality_checks)

,check_name,issue_count
0,accounts,0
1,users.account_id -> accounts,0
2,usage_events.user_id -> users,0
3,subscriptions.account_id -> accounts,0
4,subscriptions.plan_id -> plans,0
5,invoices.subscription_id -> subscriptions,0


In [ ]:
warehouse.execute("CREATE SCHEMA IF NOT EXISTS mart")

print("Analytical schema created.")

Analytical schema created.


In [ ]:
warehouse.execute("""
    CREATE OR REPLACE TABLE mart.dim_account AS
    SELECT
        account_id,
        account_name,
        industry,
        CAST(created_at AS DATE) AS account_created_date
    FROM raw.accounts;

    CREATE OR REPLACE TABLE mart.dim_user AS
    SELECT
        user_id,
        account_id,
        email,
        CAST(created_at AS DATE) AS user_created_date
    FROM raw.users;

    CREATE OR REPLACE TABLE mart.dim_plan AS
    SELECT
        plan_id,
        plan_name,
        monthly_price
    FROM raw.plans;

    CREATE OR REPLACE TABLE mart.dim_subscription AS
    SELECT
        subscription_id,
        account_id,
        plan_id,
        CAST(start_date AS DATE) AS start_date,
        CAST(end_date AS DATE) AS end_date,
        status
    FROM raw.subscriptions;
""")

for table in ["dim_account", "dim_user", "dim_plan", "dim_subscription"]:
    count = warehouse.execute(
        f"SELECT COUNT(*) FROM mart.{table}"
    ).fetchone()[0]

    print(f"mart.{table}: {count:,} records")

mart.dim_account: 500 records
mart.dim_user: 3,000 records
mart.dim_plan: 3 records
mart.dim_subscription: 500 records


In [ ]:
warehouse.execute("""
    CREATE OR REPLACE TABLE mart.fact_usage AS
    SELECT
        e.event_id,
        u.account_id,
        e.user_id,
        CAST(e.event_timestamp AS TIMESTAMP) AS event_timestamp,
        CAST(e.event_timestamp AS DATE) AS event_date,
        e.feature_name,
        e.event_type,
        e.quantity
    FROM raw.usage_events e
    JOIN mart.dim_user u
        ON e.user_id = u.user_id;

    CREATE OR REPLACE TABLE mart.fact_invoice AS
    SELECT
        i.invoice_id,
        s.subscription_id,
        s.account_id,
        s.plan_id,
        CAST(i.invoice_date AS DATE) AS invoice_date,
        i.amount,
        i.payment_status,
        CAST(i.paid_at AS DATE) AS paid_date
    FROM raw.invoices i
    JOIN mart.dim_subscription s
        ON i.subscription_id = s.subscription_id;
""")

for table in ["fact_usage", "fact_invoice"]:
    count = warehouse.execute(
        f"SELECT COUNT(*) FROM mart.{table}"
    ).fetchone()[0]

    print(f"mart.{table}: {count:,} records")

mart.fact_usage: 50,000 records
mart.fact_invoice: 2,749 records


In [ ]:
fact_checks = warehouse.execute("""
    SELECT
        'fact_usage' AS table_name,
        COUNT(*) AS total_rows,
        COUNT(DISTINCT event_id) AS unique_ids,
        COUNT(*) - COUNT(DISTINCT event_id) AS duplicate_ids
    FROM mart.fact_usage

    UNION ALL

    SELECT
        'fact_invoice',
        COUNT(*),
        COUNT(DISTINCT invoice_id),
        COUNT(*) - COUNT(DISTINCT invoice_id)
    FROM mart.fact_invoice
""").df()

display(fact_checks)

,table_name,total_rows,unique_ids,duplicate_ids
0,fact_usage,50000,50000,0
1,fact_invoice,2749,2749,0


In [ ]:
warehouse.execute("""
    CREATE OR REPLACE TABLE mart.customer_metrics AS

    WITH usage_summary AS (
        SELECT
            account_id,
            COUNT(*) AS total_usage_events,
            MAX(event_date) AS last_activity_date,
            SUM(
                CASE
                    WHEN event_date >= DATE '2026-06-01'
                    THEN 1 ELSE 0
                END
            ) AS usage_events_last_30_days
        FROM mart.fact_usage
        GROUP BY account_id
    ),

    invoice_summary AS (
        SELECT
            account_id,
            COUNT(*) AS total_invoices,
            SUM(
                CASE WHEN payment_status = 'PAID'
                     THEN amount ELSE 0 END
            ) AS total_paid_revenue,
            SUM(
                CASE WHEN payment_status = 'FAILED'
                     THEN 1 ELSE 0 END
            ) AS failed_invoice_count,
            SUM(
                CASE WHEN payment_status = 'PENDING'
                     THEN 1 ELSE 0 END
            ) AS pending_invoice_count
        FROM mart.fact_invoice
        GROUP BY account_id
    )

    SELECT
        a.account_id,
        a.account_name,
        a.industry,
        s.subscription_id,
        s.status AS subscription_status,
        p.plan_name,
        p.monthly_price,
        CASE
            WHEN s.status = 'ACTIVE' THEN p.monthly_price
            ELSE 0
        END AS mrr,
        COALESCE(u.total_usage_events, 0) AS total_usage_events,
        u.last_activity_date,
        COALESCE(u.usage_events_last_30_days, 0)
            AS usage_events_last_30_days,
        COALESCE(i.total_invoices, 0) AS total_invoices,
        COALESCE(i.total_paid_revenue, 0) AS total_paid_revenue,
        COALESCE(i.failed_invoice_count, 0) AS failed_invoice_count,
        COALESCE(i.pending_invoice_count, 0) AS pending_invoice_count

    FROM mart.dim_account a
    LEFT JOIN mart.dim_subscription s
        ON a.account_id = s.account_id
    LEFT JOIN mart.dim_plan p
        ON s.plan_id = p.plan_id
    LEFT JOIN usage_summary u
        ON a.account_id = u.account_id
    LEFT JOIN invoice_summary i
        ON a.account_id = i.account_id
""")

print("Customer-level analytics table created.")

display(
    warehouse.execute("""
        SELECT *
        FROM mart.customer_metrics
        ORDER BY account_id
        LIMIT 5
    """).df()
)

Customer-level analytics table created.


,account_id,account_name,industry,subscription_id,subscription_status,plan_name,monthly_price,mrr,total_usage_events,last_activity_date,usage_events_last_30_days,total_invoices,total_paid_revenue,failed_invoice_count,pending_invoice_count
0,1,Company_0001,Retail,1,ACTIVE,Enterprise,3999.0,3999.0,87,2026-06-29,17.0,10,35991.0,0.0,1.0
1,2,Company_0002,Education,2,PAST_DUE,Basic,499.0,0.0,89,2026-06-29,10.0,7,2994.0,0.0,1.0
2,3,Company_0003,Healthcare,3,ACTIVE,Basic,499.0,499.0,151,2026-06-29,23.0,7,3493.0,0.0,0.0
3,4,Company_0004,Education,4,ACTIVE,Basic,499.0,499.0,109,2026-06-30,24.0,4,1996.0,0.0,0.0
4,5,Company_0005,Education,5,ACTIVE,Professional,1499.0,1499.0,123,2026-06-30,18.0,10,11992.0,1.0,1.0


In [ ]:
revenue_summary = warehouse.execute("""
    SELECT
        COUNT(*) AS total_customers,
        COUNT(*) FILTER (
            WHERE subscription_status = 'ACTIVE'
        ) AS active_customers,
        COUNT(*) FILTER (
            WHERE subscription_status = 'CANCELLED'
        ) AS cancelled_customers,
        COUNT(*) FILTER (
            WHERE subscription_status = 'PAST_DUE'
        ) AS past_due_customers,
        SUM(mrr) AS total_mrr,
        SUM(total_paid_revenue) AS total_collected_revenue,
        SUM(failed_invoice_count) AS failed_invoices,
        SUM(pending_invoice_count) AS pending_invoices
    FROM mart.customer_metrics
""").df()

display(revenue_summary)

,total_customers,active_customers,cancelled_customers,past_due_customers,total_mrr,total_collected_revenue,failed_invoices,pending_invoices
0,500,418,49,33,642582.0,3644075.0,207.0,117.0


In [ ]:
plan_performance = warehouse.execute("""
    SELECT
        plan_name,
        COUNT(*) AS total_customers,
        COUNT(*) FILTER (
            WHERE subscription_status = 'ACTIVE'
        ) AS active_customers,
        SUM(mrr) AS current_mrr,
        SUM(total_paid_revenue) AS collected_revenue,
        SUM(failed_invoice_count) AS failed_invoices,
        SUM(pending_invoice_count) AS pending_invoices
    FROM mart.customer_metrics
    GROUP BY plan_name
    ORDER BY current_mrr DESC
""").df()

display(plan_performance)

,plan_name,total_customers,active_customers,current_mrr,collected_revenue,failed_invoices,pending_invoices
0,Enterprise,98,80,319920.0,1775556.0,41.0,25.0
1,Professional,182,154,230846.0,1319120.0,83.0,47.0
2,Basic,220,184,91816.0,549399.0,83.0,45.0


In [ ]:
feature_adoption = warehouse.execute("""
    SELECT
        feature_name,
        COUNT(*) AS total_events,
        COUNT(DISTINCT user_id) AS active_users,
        COUNT(DISTINCT account_id) AS active_accounts,
        SUM(quantity) AS total_quantity
    FROM mart.fact_usage
    GROUP BY feature_name
    ORDER BY total_events DESC
""").df()

display(feature_adoption)

,feature_name,total_events,active_users,active_accounts,total_quantity
0,Projects,12552,2950,498,68908.0
1,Reports,10004,2912,498,55099.0
2,Dashboard,9963,2887,498,54725.0
3,Collaboration,9929,2885,497,54414.0
4,File Upload,7552,2762,496,41608.0


In [ ]:
usage_by_plan = warehouse.execute("""
    SELECT
        p.plan_name,
        COUNT(DISTINCT s.account_id) AS customer_accounts,
        COUNT(f.event_id) AS total_usage_events,
        ROUND(
            COUNT(f.event_id) * 1.0
            / COUNT(DISTINCT s.account_id),
            2
        ) AS events_per_customer,
        ROUND(
            COUNT(DISTINCT f.user_id) * 1.0
            / COUNT(DISTINCT s.account_id),
            2
        ) AS active_users_per_customer
    FROM mart.fact_usage f
    JOIN mart.dim_subscription s
        ON f.account_id = s.account_id
    JOIN mart.dim_plan p
        ON s.plan_id = p.plan_id
    GROUP BY p.plan_name
    ORDER BY events_per_customer DESC
""").df()

display(usage_by_plan)

,plan_name,customer_accounts,total_usage_events,events_per_customer,active_users_per_customer
0,Basic,220,22449,102.04,6.08
1,Professional,181,18142,100.23,6.07
2,Enterprise,97,9409,97.00,5.80


In [ ]:
warehouse.execute("""
    CREATE OR REPLACE TABLE mart.customer_health AS

    WITH recent_payment_issues AS (
        SELECT
            account_id,
            COUNT(*) AS failed_invoices_last_30_days
        FROM mart.fact_invoice
        WHERE payment_status = 'FAILED'
          AND invoice_date >= DATE '2026-06-01'
          AND invoice_date <= DATE '2026-06-30'
        GROUP BY account_id
    )

    SELECT
        c.*,
        COALESCE(p.failed_invoices_last_30_days, 0)
            AS failed_invoices_last_30_days,

        CASE
            WHEN c.subscription_status = 'ACTIVE'
             AND c.usage_events_last_30_days = 0
            THEN TRUE
            ELSE FALSE
        END AS low_engagement,

        CASE
            WHEN COALESCE(p.failed_invoices_last_30_days, 0) > 0
            THEN TRUE
            ELSE FALSE
        END AS recent_payment_concern,

        CASE
            WHEN c.subscription_status = 'ACTIVE'
             AND (
                 c.usage_events_last_30_days = 0
                 OR COALESCE(p.failed_invoices_last_30_days, 0) > 0
             )
            THEN 'AT_RISK'
            WHEN c.subscription_status = 'ACTIVE'
            THEN 'HEALTHY'
            ELSE c.subscription_status
        END AS customer_health_status

    FROM mart.customer_metrics c
    LEFT JOIN recent_payment_issues p
        ON c.account_id = p.account_id
""")

display(warehouse.execute("""
    SELECT
        customer_health_status,
        COUNT(*) AS customer_count
    FROM mart.customer_health
    GROUP BY customer_health_status
    ORDER BY customer_count DESC
""").df())

,customer_health_status,customer_count
0,HEALTHY,383
1,CANCELLED,49
2,AT_RISK,35
3,PAST_DUE,33


In [ ]:
at_risk_customers = warehouse.execute("""
    SELECT
        account_id,
        account_name,
        plan_name,
        mrr,
        last_activity_date,
        usage_events_last_30_days,
        failed_invoices_last_30_days,
        low_engagement,
        recent_payment_concern
    FROM mart.customer_health
    WHERE customer_health_status = 'AT_RISK'
    ORDER BY
        recent_payment_concern DESC,
        low_engagement DESC,
        mrr DESC
""").df()

print("At-risk customers:", len(at_risk_customers))
display(at_risk_customers.head(15))

At-risk customers: 35


,account_id,account_name,plan_name,mrr,last_activity_date,usage_events_last_30_days,failed_invoices_last_30_days,low_engagement,recent_payment_concern
0,44,Company_0044,Enterprise,3999.0,2026-06-30,16.0,1,False,True
1,162,Company_0162,Enterprise,3999.0,2026-06-29,13.0,1,False,True
2,184,Company_0184,Enterprise,3999.0,2026-06-26,5.0,1,False,True
3,241,Company_0241,Enterprise,3999.0,2026-06-28,10.0,1,False,True
4,286,Company_0286,Enterprise,3999.0,2026-06-30,21.0,1,False,True
5,294,Company_0294,Enterprise,3999.0,2026-06-25,12.0,1,False,True
6,423,Company_0423,Enterprise,3999.0,2026-06-28,6.0,1,False,True
7,41,Company_0041,Professional,1499.0,2026-06-29,8.0,1,False,True
8,47,Company_0047,Professional,1499.0,2026-06-29,23.0,1,False,True
9,63,Company_0063,Professional,1499.0,2026-06-30,15.0,1,False,True


In [ ]:
final_checks = warehouse.execute("""
    SELECT 'dim_account' AS table_name,
           COUNT(*) AS total_rows,
           COUNT(DISTINCT account_id) AS unique_ids,
           COUNT(*) FILTER (WHERE account_id IS NULL) AS null_ids
    FROM mart.dim_account

    UNION ALL

    SELECT 'dim_user', COUNT(*),
           COUNT(DISTINCT user_id),
           COUNT(*) FILTER (WHERE user_id IS NULL)
    FROM mart.dim_user

    UNION ALL

    SELECT 'dim_subscription', COUNT(*),
           COUNT(DISTINCT subscription_id),
           COUNT(*) FILTER (WHERE subscription_id IS NULL)
    FROM mart.dim_subscription

    UNION ALL

    SELECT 'fact_usage', COUNT(*),
           COUNT(DISTINCT event_id),
           COUNT(*) FILTER (WHERE event_id IS NULL)
    FROM mart.fact_usage

    UNION ALL

    SELECT 'fact_invoice', COUNT(*),
           COUNT(DISTINCT invoice_id),
           COUNT(*) FILTER (WHERE invoice_id IS NULL)
    FROM mart.fact_invoice

    UNION ALL

    SELECT 'customer_metrics', COUNT(*),
           COUNT(DISTINCT account_id),
           COUNT(*) FILTER (WHERE account_id IS NULL)
    FROM mart.customer_metrics

    UNION ALL

    SELECT 'customer_health', COUNT(*),
           COUNT(DISTINCT account_id),
           COUNT(*) FILTER (WHERE account_id IS NULL)
    FROM mart.customer_health
""").df()

final_checks["duplicate_ids"] = (
    final_checks["total_rows"] - final_checks["unique_ids"]
)

display(final_checks)

,table_name,total_rows,unique_ids,null_ids,duplicate_ids
0,dim_account,500,500,0,0
1,dim_user,3000,3000,0,0
2,dim_subscription,500,500,0,0
3,fact_usage,50000,50000,0,0
4,fact_invoice,2749,2749,0,0
5,customer_metrics,500,500,0,0
6,customer_health,500,500,0,0


In [ ]:
reports = {
    "customer_metrics": "SELECT * FROM mart.customer_metrics",
    "customer_health": "SELECT * FROM mart.customer_health",
    "plan_performance": """
        SELECT
            plan_name,
            COUNT(*) AS total_customers,
            COUNT(*) FILTER (
                WHERE subscription_status = 'ACTIVE'
            ) AS active_customers,
            SUM(mrr) AS current_mrr,
            SUM(total_paid_revenue) AS collected_revenue,
            SUM(failed_invoice_count) AS failed_invoices
        FROM mart.customer_metrics
        GROUP BY plan_name
    """
}

for report_name, query in reports.items():
    report = warehouse.execute(query).df()
    report.to_csv(PROJECT_DIR / f"{report_name}.csv", index=False)
    print(f"{report_name}.csv exported: {len(report):,} rows")

customer_metrics.csv exported: 500 rows
customer_health.csv exported: 500 rows
plan_performance.csv exported: 3 rows
